# 06 -- Interactive Lollipop Plot

A fully interactive Plotly figure showing all OTOF ClinVar variants along
the protein sequence, integrating clinical classification, gnomAD allele
frequency, molecular consequence, domain assignment, and ConSurf conservation.

## Rationale

Static lollipop plots (notebook 03) are suitable for publication figures but
limit exploration. An interactive plot lets a clinician or researcher:

- Hover over any variant to see its full annotation
- Zoom into any domain region
- Filter by classification or consequence using the legend
- Immediately identify rare pathogenic variants with large gnomAD allele frequency

The HTML export is self-contained (Plotly JS bundled inline) so it works
offline and can be shared as a single file.

In [ ]:
# Install plotly if missing
import importlib, subprocess, sys
if importlib.util.find_spec('plotly') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly'])

from pathlib import Path
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DATA_DIR    = Path('../data')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

PROTEIN_LEN = 1997

DOMAINS = [
    {'name': 'C2A', 'start': 1,    'end': 122,  'color': '#4C72B0', 'rgba': 'rgba(76,114,176,0.15)'},
    {'name': 'C2B', 'start': 360,  'end': 480,  'color': '#55A868', 'rgba': 'rgba(85,168,104,0.15)'},
    {'name': 'C2C', 'start': 481,  'end': 596,  'color': '#C44E52', 'rgba': 'rgba(196,78,82,0.15)'},
    {'name': 'C2D', 'start': 940,  'end': 1054, 'color': '#8172B2', 'rgba': 'rgba(129,114,178,0.15)'},
    {'name': 'C2E', 'start': 1158, 'end': 1273, 'color': '#CCB974', 'rgba': 'rgba(204,185,116,0.15)'},
    {'name': 'C2F', 'start': 1481, 'end': 1597, 'color': '#64B5CD', 'rgba': 'rgba(100,181,205,0.15)'},
    {'name': 'TM',  'start': 1942, 'end': 1973, 'color': '#777777', 'rgba': 'rgba(119,119,119,0.20)'},
]

print('Setup complete. Plotly version:', __import__('plotly').__version__)

Setup complete. Plotly version: 6.3.0


## 1. Load and Prepare Data Sources

We load four data sources:

1. **ClinVar** (`clinvar_result.txt`): classification, consequence, HGVS notation
2. **gnomAD** (`gnomad_otof_variants.csv`): population allele frequency
3. **ConSurf** (`consurf_Q9HC10_grades.csv`): per-residue conservation grade (if available)
4. **Domain assignment**: computed from position using the boundaries above

The join key between ClinVar and gnomAD is the protein consequence string
(e.g., 'p.Arg963Ter'), because ClinVar accession IDs and gnomAD rsIDs
do not overlap reliably in this dataset.

In [ ]:
# Load ClinVar
clinvar = pd.read_csv(DATA_DIR / 'clinvar_result.txt', sep='\t', low_memory=False)
clinvar = clinvar[clinvar['Gene(s)'].str.contains('OTOF', na=False)].copy()

# Load gnomAD
gnomad = pd.read_csv(DATA_DIR / 'gnomad_otof_variants.csv')

# Load ConSurf (may not exist if notebook 04 was not run yet)
consurf_path = DATA_DIR / 'consurf_Q9HC10_grades.csv'
if consurf_path.exists():
    consurf_df = pd.read_csv(consurf_path)
    consurf_df['position'] = consurf_df['position'].astype(int)
    consurf_lookup = consurf_df.set_index('position')['grade'].to_dict()
    print(f'ConSurf loaded: {len(consurf_df):,} residues')
else:
    consurf_lookup = {}
    print('ConSurf data not found -- run notebook 04 first, or continue without it')

print(f'ClinVar: {len(clinvar):,} rows')
print(f'gnomAD : {len(gnomad):,} rows')
print('gnomAD columns:', gnomad.columns.tolist())

ConSurf loaded: 1,997 residues
ClinVar: 2,432 rows
gnomAD : 9,601 rows
gnomAD columns: ['gnomAD ID', 'Chromosome', 'Position', 'rsIDs', 'Reference', 'Alternate', 'Source', 'Filters - exomes', 'Filters - genomes', 'Transcript', 'HGVS Consequence', 'Protein Consequence', 'Transcript Consequence', 'VEP Annotation', 'ClinVar Germline Classification', 'ClinVar Variation ID', 'Flags', 'Allele Count', 'Allele Number', 'Allele Frequency', 'Homozygote Count', 'Hemizygote Count', 'Filters - joint', 'GroupMax FAF group', 'GroupMax FAF frequency', 'cadd', 'revel_max', 'spliceai_ds_max', 'pangolin_largest_ds', 'phylop', 'sift_max', 'polyphen_max', 'Allele Count African/African American', 'Allele Number African/African American', 'Homozygote Count African/African American', 'Hemizygote Count African/African American', 'Allele Count Admixed American', 'Allele Number Admixed American', 'Homozygote Count Admixed American', 'Hemizygote Count Admixed American', 'Allele Count Ashkenazi Jewish', 'Allele Nu

/var/folders/v2/52hczvlj4c9b366t1jxbm4sw0000gp/T/ipykernel_69969/1742073892.py:6: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  gnomad = pd.read_csv(DATA_DIR / 'gnomad_otof_variants.csv')


## 2. Extract Amino Acid Positions and Assign Domains

In [ ]:
# Parse aa position from HGVS notation
AA_PATTERN = re.compile(r'p[.]([A-Za-z]{3})(\d+)', re.IGNORECASE)

def extract_aa_pos(name):
    if pd.isna(name): return np.nan
    m = AA_PATTERN.search(str(name))
    return int(m.group(2)) if m else np.nan

def assign_domain(pos):
    if pd.isna(pos): return 'linker'
    for d in DOMAINS:
        if d['start'] <= pos <= d['end']:
            return d['name']
    return 'linker'

clinvar['aa_position'] = clinvar['Name'].apply(extract_aa_pos)
clinvar['domain']      = clinvar['aa_position'].apply(assign_domain)

# Simplified classification
def simplify_class(cls):
    if pd.isna(cls): return 'Other'
    cls = str(cls).strip()
    if cls in ('Pathogenic', 'Likely pathogenic', 'Pathogenic/Likely pathogenic'):
        return 'P/LP'
    if cls == 'Uncertain significance': return 'VUS'
    if cls in ('Benign', 'Likely benign', 'Benign/Likely benign'): return 'Benign'
    if 'Conflicting' in cls: return 'Conflicting'
    return 'Other'

clinvar['simple_class'] = clinvar['Germline classification'].apply(simplify_class)

# Molecular consequence category
def consequence_cat(mc):
    if pd.isna(mc): return 'other'
    mc = str(mc).lower()
    if 'missense' in mc: return 'missense'
    if 'frameshift' in mc: return 'frameshift'
    if 'splice' in mc: return 'splice'
    if 'stop_gained' in mc or 'nonsense' in mc: return 'nonsense'
    return 'other'

clinvar['consequence'] = clinvar['Molecular consequence'].apply(consequence_cat)

print(f'Variants with aa position: {clinvar["aa_position"].notna().sum():,}')
print(clinvar['simple_class'].value_counts())
print(clinvar['consequence'].value_counts())

Variants with aa position: 1,600
simple_class
Benign         1277
VUS             499
P/LP            390
Conflicting     231
Other            35
Name: count, dtype: int64
consequence
other         1414
missense       698
frameshift     115
splice         104
nonsense       101
Name: count, dtype: int64


## 3. Join gnomAD Allele Frequency

gnomAD variants are matched to ClinVar by protein consequence string,
which is the most reliable common key. Where multiple gnomAD entries
share the same protein consequence, we take the maximum allele frequency
(the most common population-level observation).

In [ ]:
# Normalise protein consequence for joining
gnomad_af_col = 'Allele Frequency'
gnomad_prot_col = 'Protein Consequence'

# Confirm columns exist
print('gnomAD AF column:', gnomad_af_col in gnomad.columns)
print('gnomAD protein col:', gnomad_prot_col in gnomad.columns)

gnomad_agg = (
    gnomad
    .dropna(subset=[gnomad_prot_col, gnomad_af_col])
    .groupby(gnomad_prot_col)[gnomad_af_col]
    .max()
    .reset_index()
    .rename(columns={gnomad_prot_col: 'Protein change', gnomad_af_col: 'gnomad_af'})
)

clinvar = clinvar.merge(gnomad_agg, on='Protein change', how='left')
print(f'Variants with gnomAD AF: {clinvar["gnomad_af"].notna().sum():,}')

# Map ConSurf grade
clinvar['consurf_grade'] = clinvar['aa_position'].map(consurf_lookup)
print(f'Variants with ConSurf: {clinvar["consurf_grade"].notna().sum():,}')

gnomAD AF column: True
gnomAD protein col: True
Variants with gnomAD AF: 0
Variants with ConSurf: 1,600


## 4. Compute Y-axis Positions

Y-axis layout:
- P/LP: y = +1
- Conflicting: y = +0.5
- VUS: y = 0
- Benign: y = -1
- Other: y = -0.5

Small jitter is added within each band to avoid overplotting.

In [ ]:
rng = np.random.default_rng(123)

Y_BASE = {'P/LP': 1.0, 'Conflicting': 0.5, 'VUS': 0.0, 'Other': -0.5, 'Benign': -1.0}
JITTER_SCALE = 0.12

clinvar_plot = clinvar.dropna(subset=['aa_position']).copy()
clinvar_plot['y_base']  = clinvar_plot['simple_class'].map(Y_BASE)
clinvar_plot['y_jitter'] = clinvar_plot['y_base'] + rng.uniform(
    -JITTER_SCALE, JITTER_SCALE, size=len(clinvar_plot)
)

# Marker size: proportional to log10(gnomAD AF + 1e-8)
# Scaled so AF=0 -> size ~6, AF=0.001 -> size ~18
def af_to_size(af):
    if pd.isna(af) or af == 0:
        return 6
    return max(6, 6 + 20 * (np.log10(af + 1e-8) + 8) / 8)

clinvar_plot['marker_size'] = clinvar_plot['gnomad_af'].apply(af_to_size)

print(f'Plotting {len(clinvar_plot):,} variants')
print('Y base distribution:')
print(clinvar_plot['simple_class'].value_counts())

Plotting 1,600 variants
Y base distribution:
simple_class
Benign         678
VUS            440
P/LP           255
Conflicting    201
Other           26
Name: count, dtype: int64


## 5. Build Interactive Plotly Figure

Each trace corresponds to one (classification, consequence) pair so that
Plotly's legend allows toggling by both dimensions. Domain bands are added
as filled rectangles in the background.

In [ ]:
# Colour and shape encoding
CLASS_COLORS = {
    'P/LP':        '#d62728',
    'VUS':         '#9467bd',
    'Benign':      '#2ca02c',
    'Conflicting': '#ff7f0e',
    'Other':       '#aaaaaa',
}
CONSQ_SYMBOLS = {
    'missense':   'circle',
    'frameshift': 'triangle-up',
    'splice':     'square',
    'nonsense':   'diamond',
    'other':      'cross',
}

fig = go.Figure()

# Domain background bands
for d in DOMAINS:
    mid = (d['start'] + d['end']) / 2
    fig.add_shape(
        type='rect',
        x0=d['start'], x1=d['end'], y0=-1.4, y1=1.4,
        fillcolor=d['rgba'],
        line=dict(width=0),
        layer='below'
    )
    fig.add_annotation(
        x=mid, y=1.38,
        text=f"<b>{d['name']}</b>",
        showarrow=False,
        font=dict(size=10, color=d['color']),
        yanchor='top'
    )

# Domain boundary lines
for d in DOMAINS:
    for x_pos in [d['start'], d['end']]:
        fig.add_shape(
            type='line',
            x0=x_pos, x1=x_pos, y0=-1.3, y1=1.3,
            line=dict(color=d['color'], width=0.8, dash='dot'),
            layer='below'
        )

# Y-axis band labels
for label, y in [('P/LP', 1.0), ('Conflicting', 0.5), ('VUS', 0.0),
                  ('Other', -0.5), ('Benign', -1.0)]:
    fig.add_annotation(
        x=PROTEIN_LEN + 10, y=y,
        text=f'<i>{label}</i>',
        showarrow=False,
        font=dict(size=9, color=CLASS_COLORS.get(label, '#444')),
        xanchor='left'
    )

# Variant traces: one per (class, consequence) combination
for cls in ['P/LP', 'Conflicting', 'VUS', 'Other', 'Benign']:
    for consq in ['missense', 'frameshift', 'splice', 'nonsense', 'other']:
        subset = clinvar_plot[
            (clinvar_plot['simple_class'] == cls) &
            (clinvar_plot['consequence']  == consq)
        ]
        if len(subset) == 0:
            continue
        hover_texts = []
        for _, row in subset.iterrows():
            af_str  = f"{row['gnomad_af']:.2e}" if pd.notna(row['gnomad_af']) else 'not in gnomAD'
            cs_str  = str(int(row['consurf_grade'])) if pd.notna(row['consurf_grade']) else 'N/A'
            hgvs    = str(row['Name'])[:80]
            hover_texts.append(
                f"<b>OTOF</b> {row.get('Protein change', hgvs)}<br>"
                f"Class: {row['Germline classification']}<br>"
                f"Consequence: {row['Molecular consequence']}<br>"
                f"Domain: {row['domain']}<br>"
                f"gnomAD AF: {af_str}<br>"
                f"ConSurf: {cs_str}<br>"
                f"Position: {int(row['aa_position']) if pd.notna(row['aa_position']) else 'N/A'}"
            )
        fig.add_trace(go.Scatter(
            x=subset['aa_position'],
            y=subset['y_jitter'],
            mode='markers',
            name=f'{cls} / {consq}',
            marker=dict(
                symbol=CONSQ_SYMBOLS[consq],
                color=CLASS_COLORS[cls],
                size=subset['marker_size'],
                opacity=0.80,
                line=dict(width=0.5, color='rgba(0,0,0,0.3)'),
            ),
            text=hover_texts,
            hovertemplate='%{text}<extra></extra>',
            legendgroup=cls,
            legendgrouptitle_text=cls if consq == 'missense' else None,
        ))

print('Traces added:', len(fig.data))

Traces added: 15


In [ ]:
fig.update_layout(
    title=dict(
        text=('<b>OTOF Variant Landscape</b><br>'
              '<sub>ClinVar classifications | gnomAD allele frequency (point size) | '
              'Molecular consequence (shape) | Hover for full annotation</sub>'),
        x=0.5, xanchor='center', font=dict(size=16)
    ),
    xaxis=dict(
        title='Amino acid position (otoferlin, 1,997 aa)',
        range=[0, PROTEIN_LEN + 60],
        showgrid=True, gridcolor='#eeeeee',
    ),
    yaxis=dict(
        title='Clinical classification',
        tickvals=[-1, -0.5, 0, 0.5, 1],
        ticktext=['Benign', 'Other', 'VUS', 'Conflicting', 'P/LP'],
        range=[-1.5, 1.5],
        showgrid=True, gridcolor='#eeeeee',
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=650,
    width=1400,
    legend=dict(
        title='Classification / Consequence',
        groupclick='toggleitem',
        x=1.01, y=1,
        bordercolor='#cccccc',
        borderwidth=1,
    ),
    margin=dict(r=220),
    hovermode='closest',
)

fig.show()
print('Figure rendered.')

Figure rendered.


## 6. Export Static PNG

The interactive figure displays inline in the notebook via Plotly.
A static PNG is saved to `results/` for inclusion in documents and presentations.
The notebook itself is the interactive artifact -- open it in Jupyter to explore the data.

In [ ]:
# Static PNG via Kaleido
png_path = RESULTS_DIR / 'otof_interactive_lollipop_static.png'
try:
    pio.write_image(fig, str(png_path), format='png',
                    width=1400, height=650, scale=2)
    print(f'Saved PNG: {png_path}')
except Exception as e:
    print(f'PNG export failed ({e}). Install kaleido: pip install kaleido')
    # Fallback: matplotlib static version
    import matplotlib.patches as mpatches
    fig_mpl, ax = plt.subplots(figsize=(16, 6))
    for d in DOMAINS:
        ax.axvspan(d['start'], d['end'], alpha=0.12, color=d['color'])
        mid = (d['start'] + d['end']) / 2
        ax.text(mid, 1.45, d['name'], ha='center', fontsize=8,
                color=d['color'], fontweight='bold')
    for cls, color in CLASS_COLORS.items():
        sub = clinvar_plot[clinvar_plot['simple_class'] == cls]
        ax.scatter(sub['aa_position'], sub['y_jitter'],
                   c=color, s=sub['marker_size'] * 1.5,
                   alpha=0.7, label=cls, edgecolors='none')
    ax.set_xlim(0, PROTEIN_LEN)
    ax.set_ylim(-1.5, 1.55)
    ax.set_yticks([-1, -0.5, 0, 0.5, 1])
    ax.set_yticklabels(['Benign', 'Other', 'VUS', 'Conflicting', 'P/LP'])
    ax.set_xlabel('Amino acid position', fontsize=11)
    ax.set_ylabel('Classification', fontsize=11)
    ax.set_title('OTOF Variant Landscape', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved PNG (matplotlib fallback): {png_path}')

In [ ]:
print('=' * 60)
print('Interactive Lollipop -- Summary')
print('=' * 60)
print(f'Variants plotted          : {len(clinvar_plot):,}')
print(f'With gnomAD AF            : {clinvar_plot["gnomad_af"].notna().sum():,}')
print(f'With ConSurf score        : {clinvar_plot["consurf_grade"].notna().sum():,}')
print(f'Static PNG                : results/otof_interactive_lollipop_static.png')
print('Open this notebook in Jupyter for the interactive version.')
print('=' * 60)